In [ ]:
import json
import os
import sys

import tensorflow as tf

sys.path.append(os.path.abspath("../"))

from util import camera as u_camera
from util import dataset_io as u_dataset_io
from util import metrics as u_metrics

In [ ]:
test_groundtruth_path = "../../data/selected/test_groundtruth.json"
test_bhuman_path = "../../data/selected/test_b-human.json"
test_model_path = "../../data/selected/test_model.json"

test_groundtruth = None
test_bhuman = None
test_model = None

with open(test_groundtruth_path) as f:
    test_groundtruth = json.load(f)
with open(test_bhuman_path) as f:
    test_bhuman = json.load(f)
with open(test_model_path) as f:
    test_model = json.load(f)

assert len(test_groundtruth) == len(test_bhuman)
assert len(test_bhuman) == len(test_model)

## Intersections


In [ ]:
def extract_coordinates(intersections):
    # Convert list of dicts to list of lists: [[x1, y1], [x2, y2], ...]
    return [[coord["x"], coord["y"]] for coord in intersections]

In [ ]:
def filter_coords_by_distance(coords_tensor, camera, intrinsics, max_distance: float):
    if tf.shape(coords_tensor)[0] > 0:
        bhuman_distances = tf.linalg.norm(
            u_camera.image_to_world(camera, intrinsics, coords_tensor), axis=-1, keepdims=True
        )
        return tf.boolean_mask(coords_tensor, tf.squeeze(bhuman_distances <= max_distance, axis=-1))
    else:
        return coords_tensor

In [ ]:
threshold = 15.0
max_distance = 5

model_true_positives = {"L": 0, "T": 0, "X": 0}
model_false_negatives = {"L": 0, "T": 0, "X": 0}
model_false_positives = {"L": 0, "T": 0, "X": 0}

bhuman_true_positives = {"L": 0, "T": 0, "X": 0}
bhuman_false_negatives = {"L": 0, "T": 0, "X": 0}
bhuman_false_positives = {"L": 0, "T": 0, "X": 0}

# Iterate over each frame
for idx, gt_frame in enumerate(test_groundtruth):
    if "intersections" not in gt_frame:
        continue
    if gt_frame["intersections"]["ignore_sample"]:
        continue

    frame_time = gt_frame["frame_time"]

    camera = u_dataset_io.camera_from_label(gt_frame)
    intrinsics = u_dataset_io.intrinsics_from_label(gt_frame)

    assert frame_time == test_model[idx]["frame_time"]
    assert frame_time == test_bhuman[idx]["frame_time"]
    # Iterate over each intersection type: 'L', 'T', 'X'
    for intersection_type in ["L", "T", "X"]:
        bhuman_coords = extract_coordinates(test_bhuman[idx]["intersections"][intersection_type])
        model_coords = extract_coordinates(test_model[idx]["intersections"][intersection_type])
        gt_coords = extract_coordinates(gt_frame["intersections"][intersection_type])

        bhuman_tensor = tf.constant(bhuman_coords, dtype=tf.float32)  # (N, 2)
        model_tensor = tf.constant(model_coords, dtype=tf.float32)  # (N, 2)
        gt_tensor = tf.constant(gt_coords, dtype=tf.float32)  # (N, 2)

        bhuman_tensor = filter_coords_by_distance(bhuman_tensor, camera, intrinsics, max_distance)
        model_tensor = filter_coords_by_distance(model_tensor, camera, intrinsics, max_distance)
        gt_tensor = filter_coords_by_distance(gt_tensor, camera, intrinsics, max_distance)

        model_matches = u_metrics.match_keypoints_image(model_tensor, gt_tensor, threshold)
        bhuman_matches = u_metrics.match_keypoints_image(bhuman_tensor, gt_tensor, threshold)

        # if model_matches["false_positives"] > 0 and len(gt_coords) > 0:
        #     tf.print("Model: ", model_tensor)
        #     tf.print("GT: ", gt_tensor)
        #     tf.print("=====")

        model_true_positives[intersection_type] += int(model_matches["true_positives"])
        model_false_negatives[intersection_type] += int(model_matches["false_negatives"])
        model_false_positives[intersection_type] += int(model_matches["false_positives"])

        bhuman_true_positives[intersection_type] += int(bhuman_matches["true_positives"])
        bhuman_false_negatives[intersection_type] += int(bhuman_matches["false_negatives"])
        bhuman_false_positives[intersection_type] += int(bhuman_matches["false_positives"])


#### Confusion Matrix


In [ ]:
print("=== Model ===")
print("TP: ", model_true_positives)
print("FP: ", model_false_positives)
print("FN: ", model_false_negatives)

print("=== B-Human ===")
print("TP : ", bhuman_true_positives)
print("FP: ", bhuman_false_positives)
print("FN: ", bhuman_false_negatives)